# P 管制圖：計數型管制圖

> **課程定位**：基本工具篇（4/6）  
> **前置課程**：[03_個別值管制圖_I-MR](./03_個別值管制圖_I-MR.ipynb)  
> **學習路徑**：01 概論 → 02 X-bar & R → 03 I-MR → **04 P Chart** → 05 製程能力 → 06 綜合案例

## 學習目標
- 區分計量型與計數型管制圖的適用場景
- 掌握 P Chart 的公式推導與計算
- 處理變動樣本大小的管制界限
- 將 P Chart 結果連結到製程改善（Pareto 分析）

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'Microsoft JhengHei', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

## 1. 計量型 vs. 計數型管制圖

| 特性 | 計量型（Variable） | 計數型（Attribute） |
|------|---------------------|---------------------|
| 數據性質 | 連續型（可量測） | 離散型（計數/分類） |
| 範例 | 長度、重量、溫度 | 不良品數、缺陷數 |
| 資訊量 | 較多（精確數值） | 較少（僅合格/不合格） |
| 適用圖表 | X-bar & R, I-MR | **P 圖**、np 圖、C 圖、u 圖 |
| 取樣成本 | 通常較高 | 通常較低 |

### 計數型管制圖的選擇

| 圖表 | 監控對象 | 樣本大小 |
|------|----------|----------|
| **P 圖** | 不良品**比例** | 可變動 |
| np 圖 | 不良品**數量** | 需固定 |
| C 圖 | 缺陷**數量** | 需固定檢驗範圍 |
| u 圖 | 缺陷**密度** | 可變動 |

本課程聚焦於最常用的 **P 圖（Proportion Chart）**。

## 2. P Chart 公式推導

### 基本定義

第 $i$ 個子組的不良率：
$$p_i = \frac{d_i}{n_i}$$

其中 $d_i$ 為不良品數，$n_i$ 為檢驗數量。

### 平均不良率

$$\bar{p} = \frac{\sum_{i=1}^{k} d_i}{\sum_{i=1}^{k} n_i}$$

### 管制界限

$$UCL_i = \bar{p} + 3\sqrt{\frac{\bar{p}(1-\bar{p})}{n_i}}$$

$$LCL_i = \max\left(0, \;\bar{p} - 3\sqrt{\frac{\bar{p}(1-\bar{p})}{n_i}}\right)$$

> **知識補給站**  
> 注意：當 $n_i$ 不同時，UCL 和 LCL 會**隨每個子組變化**。LCL 不能為負值（比例 >= 0）。

In [ ]:
def plot_p_chart(defects, sample_sizes, title="P 管制圖", labels=None):
    """
    繪製 P 管制圖
    
    Parameters:
    -----------
    defects : array-like - 每個子組的不良品數
    sample_sizes : array-like - 每個子組的檢驗數量
    title : str - 圖表標題
    labels : list, optional - 子組標籤
    """
    defects = np.array(defects, dtype=float)
    sample_sizes = np.array(sample_sizes, dtype=float)
    k = len(defects)
    
    # 計算不良率
    p = defects / sample_sizes
    p_bar = defects.sum() / sample_sizes.sum()
    
    # 管制界限（每個子組可能不同）
    UCL = p_bar + 3 * np.sqrt(p_bar * (1 - p_bar) / sample_sizes)
    LCL = np.maximum(0, p_bar - 3 * np.sqrt(p_bar * (1 - p_bar) / sample_sizes))
    
    x_axis = range(1, k + 1)
    if labels is None:
        labels = x_axis
    
    fig, ax = plt.subplots(figsize=(14, 7))
    
    # 不良率
    ooc = (p > UCL) | (p < LCL)
    ax.plot(x_axis, p, 'b-o', markersize=5, label=f'不良率 (p)', zorder=3)
    if ooc.any():
        ax.scatter(np.where(ooc)[0] + 1, p[ooc],
                   color='red', s=100, zorder=5, label='超出管制界限', marker='s')
    
    # 管制線
    ax.axhline(y=p_bar, color='green', linewidth=2, label=f'p\u0304 = {p_bar:.4f}')
    
    # 處理等/不等樣本大小
    if np.all(sample_sizes == sample_sizes[0]):
        # 等樣本大小：水平線
        ax.axhline(y=UCL[0], color='red', linestyle='--', linewidth=1.5, 
                    label=f'UCL = {UCL[0]:.4f}')
        ax.axhline(y=LCL[0], color='red', linestyle='--', linewidth=1.5, 
                    label=f'LCL = {LCL[0]:.4f}')
    else:
        # 不等樣本大小：階梯線
        ax.step(x_axis, UCL, color='red', linestyle='--', linewidth=1.5, 
                where='mid', label='UCL (變動)')
        ax.step(x_axis, LCL, color='red', linestyle='--', linewidth=1.5, 
                where='mid', label='LCL (變動)')
    
    ax.set_title(title, fontsize=16, fontweight='bold')
    ax.set_xlabel('子組編號', fontsize=12)
    ax.set_ylabel('不良率 (p)', fontsize=12)
    ax.legend(loc='upper right', fontsize=10)
    ax.grid(True, alpha=0.3)
    
    # 標註不良率百分比
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.1%}'))
    
    plt.tight_layout()
    plt.show()
    
    return {
        'p': p, 'p_bar': p_bar,
        'UCL': UCL, 'LCL': LCL,
        'ooc': np.where(ooc)[0] + 1
    }

## 3. 實務案例一：SMT 焊接不良率（等樣本大小）

### 情境描述

某電子廠的 SMT（表面黏著技術）產線，每日檢驗 **200 片** PCB 板的焊接品質。
- 收集 **25 天** 的數據
- 平均不良率約 **3%**
- 第 18 天因錫膏品質異常，不良率突然上升

In [ ]:
np.random.seed(42)

n_days = 25
sample_size = 200

# 模擬不良品數（二項分布）
defect_counts = np.random.binomial(n=sample_size, p=0.03, size=n_days)

# 第 18 天注入異常（錫膏品質問題）
defect_counts[17] = np.random.binomial(n=sample_size, p=0.10, size=1)[0]

sample_sizes_equal = np.full(n_days, sample_size)

# 顯示數據摘要
df_smt = pd.DataFrame({
    '天次': range(1, n_days + 1),
    '檢驗數量': sample_sizes_equal,
    '不良品數': defect_counts,
    '不良率': defect_counts / sample_size
})
print("SMT 焊接品質數據（前 10 天）")
display(df_smt.head(10).style.format({'不良率': '{:.2%}'}))

print(f"\n總檢驗: {sample_sizes_equal.sum()} 片")
print(f"總不良: {defect_counts.sum()} 片")
print(f"平均不良率: {defect_counts.sum() / sample_sizes_equal.sum():.2%}\n")

# 繪製 P 圖
result_smt = plot_p_chart(defect_counts, sample_sizes_equal, 
                           title="SMT 焊接不良率 P 管制圖")

print(f"超出管制界限的天次: {list(result_smt['ooc'])}")

### 案例解讀

- 第 18 天明顯超出 UCL → 錫膏品質異常
- 其餘天次的不良率在管制範圍內波動 → 屬於普通原因變異
- **行動**：調查第 18 天的錫膏批次、回焊爐溫度曲線

> **實務技巧**：P 圖偵測到異常後，下一步通常是做 **Pareto 分析** —— 分析不良品的缺陷類型分布，找出最主要的問題。

## 4. 變動樣本大小的 P Chart

在實際生產中，每天的檢驗數量可能因排程、訂單量而不同。此時 UCL/LCL 會隨 $n_i$ 變動。

In [ ]:
np.random.seed(2024)

n_days2 = 20

# 變動樣本大小（150 ~ 300）
sample_sizes_var = np.random.choice([150, 200, 250, 300], size=n_days2)

# 模擬不良品數
defect_counts_var = np.array([
    np.random.binomial(n=n, p=0.04, size=1)[0] for n in sample_sizes_var
])

# 第 12 天注入異常
defect_counts_var[11] = np.random.binomial(n=sample_sizes_var[11], p=0.12, size=1)[0]

df_var = pd.DataFrame({
    '天次': range(1, n_days2 + 1),
    '檢驗數量': sample_sizes_var,
    '不良品數': defect_counts_var,
    '不良率': defect_counts_var / sample_sizes_var
})
print("變動樣本大小數據")
display(df_var.style.format({'不良率': '{:.2%}'}))

# 繪製 P 圖
print()
result_var = plot_p_chart(defect_counts_var, sample_sizes_var,
                           title="變動樣本大小 P 管制圖")

print(f"\n注意：UCL/LCL 隨樣本大小變動")
print(f"最小 UCL: {result_var['UCL'].min():.4f} (n={sample_sizes_var[result_var['UCL'].argmin()]})")
print(f"最大 UCL: {result_var['UCL'].max():.4f} (n={sample_sizes_var[result_var['UCL'].argmax()]})")

### 觀察重點

當樣本大小 $n_i$ 變動時：
- **n 越大** → 管制界限越窄（標準誤越小）→ 偵測力越強
- **n 越小** → 管制界限越寬 → 較難偵測小的偏移

因此同一個不良率在不同天可能「在管制內」或「超出管制」——取決於當天的檢驗數量。

In [ ]:
# Pareto 分析：缺陷類型分布
np.random.seed(42)

defect_types = {
    '焊接橋接': 45,
    '焊點不足': 30,
    '偏位': 18,
    '虛焊': 12,
    '零件缺失': 8,
    '極性反轉': 5,
    '其他': 3
}

df_pareto = pd.DataFrame({
    '缺陷類型': list(defect_types.keys()),
    '次數': list(defect_types.values())
}).sort_values('次數', ascending=False).reset_index(drop=True)

df_pareto['累計百分比'] = df_pareto['次數'].cumsum() / df_pareto['次數'].sum() * 100

fig, ax1 = plt.subplots(figsize=(12, 6))

# 長條圖
bars = ax1.bar(df_pareto['缺陷類型'], df_pareto['次數'], color='steelblue', alpha=0.8, edgecolor='white')
ax1.set_ylabel('缺陷次數', fontsize=12, color='steelblue')
ax1.tick_params(axis='y', labelcolor='steelblue')

# 累計百分比線
ax2 = ax1.twinx()
ax2.plot(df_pareto['缺陷類型'], df_pareto['累計百分比'], 'r-o', linewidth=2, markersize=6)
ax2.axhline(y=80, color='orange', linestyle='--', alpha=0.7, label='80% 線')
ax2.set_ylabel('累計百分比 (%)', fontsize=12, color='red')
ax2.tick_params(axis='y', labelcolor='red')
ax2.set_ylim(0, 105)

ax1.set_title('SMT 焊接缺陷 Pareto 分析', fontsize=16, fontweight='bold')
ax1.set_xlabel('缺陷類型', fontsize=12)
ax2.legend(loc='center right', fontsize=10)

plt.tight_layout()
plt.show()

print("Pareto 分析結果：")
display(df_pareto)
print(f"\n結論：「焊接橋接」和「焊點不足」佔總缺陷的 {df_pareto.iloc[:2]['次數'].sum()/df_pareto['次數'].sum()*100:.0f}%")
print("→ 優先改善這兩項缺陷，可獲得最大效益（80/20 法則）")

## 5. 本章小結

| 項目 | 內容 |
|------|------|
| 適用場景 | 計數型數據（合格/不合格），樣本大小可變動 |
| 核心公式 | p\u0304 = \u03a3d\u1d62/\u03a3n\u1d62；UCL/LCL = p\u0304 \u00b1 3\u221a(p\u0304(1-p\u0304)/n\u1d62) |
| 變動樣本 | n 不同時，管制界限隨之變化 |
| 延伸應用 | 結合 Pareto 分析找出主要缺陷類型 |
| 關鍵函數 | `plot_p_chart()` |

---

## 課堂練習

### 基礎題
1. 某產線連續 5 天的數據如下，計算 p\u0304 和每天的 UCL：
   | 天次 | 檢驗數 | 不良品數 |
   |------|--------|----------|
   | 1 | 100 | 3 |
   | 2 | 100 | 5 |
   | 3 | 100 | 2 |
   | 4 | 100 | 4 |
   | 5 | 100 | 6 |

### 進階題
2. 使用 `plot_p_chart()` 繪製上述數據，並解釋為何即使第 5 天的不良率最高（6%），仍可能在管制範圍內。

### 挑戰題
3. 實作「修正管制界限」：排除超出管制的點後，重新計算 p\u0304 和 UCL/LCL。寫一個函數 `revised_p_chart(defects, sample_sizes)` 自動執行此流程。

---

> **下一堂課**：[05_製程能力分析_Cp_Cpk](./05_製程能力分析_Cp_Cpk.ipynb) — 從「穩不穩定」進階到「夠不夠好」